In [ ]:
import numpy as np
from numpy import sin, cos, pi
from numpy.linalg import eigvals
import matplotlib.pyplot as plt

from matplotlib import cm
from matplotlib.colors import LogNorm, Normalize

plt.style.use("../rc")

In [ ]:
import sympy as sp

B_s, t_s, p_s = sp.symbols("B t p")

ff_sym = (
    (B_s / (2 * sp.pi)) * sp.sin(t_s)**2 * sp.sin(2 * p_s)
    / ( sp.cos(p_s) ** 4 + sp.sin(p_s) ** 4 + B_s * (1 + sp.sin(2 * p_s) * sp.cos(t_s)) ** 2 )
)

# symbolic derivatives
dff_sym = sp.simplify(sp.diff(ff_sym, B_s))
ddff_sym = sp.simplify(sp.diff(dff_sym, B_s))

In [ ]:
sp.init_printing()
display(ff_sym)
display(dff_sym)
display(ddff_sym)

gg = 1 + 8 * (1 + B_s) * (dff_sym + 2*B_s*ddff_sym)
display(gg)

In [ ]:
gg = gg.simplify()

In [ ]:
display(gg)

In [ ]:
# keep names used later in the cell
# Speed up gg integration by constructing real/imag parts symbolically
b_s = sp.Symbol("b", real=True, positive=True)
B_expr = (1 - sp.I * b_s) / (1 + sp.I * b_s)

ff_sym = ff_sym.subs(B_s, B_expr)
dff_sym = dff_sym.subs(B_s, B_expr)
ddff_sym = ddff_sym.subs(B_s, B_expr)

In [ ]:
ff = sp.lambdify((b_s, t_s, p_s), ff_sym, "numpy")
dff = sp.lambdify((b_s, t_s, p_s), dff_sym, "numpy")
ddff = sp.lambdify((b_s, t_s, p_s), ddff_sym, "numpy")

In [ ]:
from scipy.integrate import dblquad
I = 1j

def integrate_tp(f_num, x, t0=0, t1=pi, p0=0, p1=pi/2):
    re_integrand = lambda p, t: np.real(f_num(x, t, p))
    im_integrand = lambda p, t: np.imag(f_num(x, t, p))
    re_val, _ = dblquad(
        re_integrand,
        t0, t1,
        lambda t: p0,
        lambda t: p1,
    )
    im_val, _ = dblquad(
        im_integrand,
        t0, t1,
        lambda t: p0,
        lambda t: p1,
    )
    val = re_val + I * im_val
    return val

In [ ]:
import json

def save_integrals(filename, b_vals, I_ff, I_dff, I_ddff):
    """Save computed integral arrays to disk as .npz file."""
    np.savez(
        filename,
        b_vals=b_vals,
        I_ff_real=np.real(I_ff),
        I_ff_imag=np.imag(I_ff),
        I_dff_real=np.real(I_dff),
        I_dff_imag=np.imag(I_dff),
        I_ddff_real=np.real(I_ddff),
        I_ddff_imag=np.imag(I_ddff),
    )
    print(f"Saved to {filename}")

def load_integrals(filename):
    """Load integral arrays from disk. Returns b_vals, I_ff, I_dff, I_ddff."""
    if not filename.endswith('.npz'):
        filename += '.npz'
    data = np.load(filename)
    b_vals = data['b_vals']
    I_ff = data['I_ff_real'] + 1j * data['I_ff_imag']
    I_dff = data['I_dff_real'] + 1j * data['I_dff_imag']
    I_ddff = data['I_ddff_real'] + 1j * data['I_ddff_imag']
    print(f"Loaded from {filename}")
    return b_vals, I_ff, I_dff, I_ddff

In [ ]:
path = "data/samples.npz"
generate_new = False

l = 20

#! Very slow...
if generate_new:
    N = 500
    b_vals = np.linspace(-l, l, 2*N + 1)
    I_ff = np.array([integrate_tp(ff, b) for b in b_vals])
    I_dff = np.array([integrate_tp(dff, b) for b in b_vals])
    I_ddff = np.array([integrate_tp(ddff, b) for b in b_vals])
    
    # Save the computed integrals
    save_integrals(path, b_vals, I_ff, I_dff, I_ddff)

In [ ]:
b_vals, I_ff, I_dff, I_ddff = load_integrals(path)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(b_vals, np.real(I_ff), color="tab:blue", label="$\\mathrm{Re} \\, f(B(b))$")
ax[0].plot(b_vals, np.real(I_dff), color="darkorange", label="$\\mathrm{Re} \\, f'(B(b))$")
ax[0].plot(b_vals, np.real(I_ddff), color="black",label="$\\mathrm{Re} \\, f''(B(b))$") 


ax[1].plot(b_vals, np.imag(I_ff), color="tab:blue", label="$\\mathrm{Im} \\, f(B(b))$")
ax[1].plot(b_vals, np.imag(I_dff), color="darkorange", label="$\\mathrm{Im} \\, f'(B(b))$")
ax[1].plot(b_vals, np.imag(I_ddff), color="black",label="$\\mathrm{Im} \\, f''(B(b))$")

ax[0].set_xlabel("$b$")
ax[1].set_xlabel("$b$")

ax[0].legend()
ax[1].legend()
plt.show()

In [ ]:
from scipy.interpolate import CubicSpline

# Create interpolating spline functions for the integral data
# Separate real and imaginary parts for interpolation

I_ff_re_spline = CubicSpline(b_vals, np.real(I_ff))
I_ff_im_spline = CubicSpline(b_vals, np.imag(I_ff))
I_ff_spline = lambda b: I_ff_re_spline(b) + 1j * I_ff_im_spline(b)

I_dff_re_spline = CubicSpline(b_vals, np.real(I_dff))
I_dff_im_spline = CubicSpline(b_vals, np.imag(I_dff))
I_dff_spline = lambda b: I_dff_re_spline(b) + 1j * I_dff_im_spline(b)

I_ddff_re_spline = CubicSpline(b_vals, np.real(I_ddff))
I_ddff_im_spline = CubicSpline(b_vals, np.imag(I_ddff))
I_ddff_spline = lambda b: I_ddff_re_spline(b) + 1j * I_ddff_im_spline(b)


In [ ]:
b_fine = np.linspace(b_vals[0], b_vals[-1], 500)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for j, (name, spline, data) in enumerate([
    ("I_ff", I_ff_spline, I_ff),
    ("I_dff", I_dff_spline, I_dff),
    ("I_ddff", I_ddff_spline, I_ddff),
]):
    axes[0, j].plot(b_fine, np.real(spline(b_fine)), '-', label='spline')
    axes[0, j].plot(b_vals, np.real(data), 'k.', markersize=3, label='data')
    axes[0, j].set_title(f"Re({name})")
    axes[0, j].legend()
    axes[0, j].set_xlabel("$b$")

    axes[1, j].plot(b_fine, np.imag(spline(b_fine)), '-', label='spline')
    axes[1, j].plot(b_vals, np.imag(data), 'k.', markersize=3, label='data')
    axes[1, j].set_title(f"Im({name})")
    axes[1, j].legend()
    axes[1, j].set_xlabel("$b$")

plt.tight_layout()
plt.show()

In [ ]:
B = lambda b : (1 - I * b) / (1 + I * b)
g = (1 + 8 * ( 1 + B(b_vals) ) * (I_dff + 2 * B(b_vals) * I_ddff) ) / (1 + I * b_vals)

g_re_spline = CubicSpline(b_vals, np.real(g))
g_im_spline = CubicSpline(b_vals, np.imag(g))
gs = lambda b: g_re_spline(b) + I * g_im_spline(b)

print(gs(0))

In [ ]:
fig, ax = plt.subplots(figsize=(6,3))

ax.plot(b_fine, np.real(gs(b_fine)), label="$\\mathrm{Re}\\,g(b)$")
ax.plot(b_fine, np.imag(gs(b_fine)), label="$\\mathrm{Im}\\,g(b)$")

ax.plot(b_vals, np.real(g), 'k.', ms=.8)
ax.plot(b_vals, np.imag(g), 'k.', ms=.8)

ax.set_ylabel("$g$")
ax.set_xlabel("$b$")
plt.legend()
# plt.show()

In [ ]:
u, a, b, c = sp.symbols("u a b c")
g1 = sp.Function("g_1")(b)
g2 = sp.Function("g_2")(b)

# h = lambda u, a, b : (u - b*a) * g1 - (a - b*u) * g2
# bb = lambda u, a, b, c : -2 * (a - b*u) * (u + h(u,a,b))

h = lambda u, a, b : u*(1 - b*a) * g1 - u*(a - b) * g2
h = lambda u, a, b : -u*(1 + b*a) * g1 - u*(a - b) * g2
bb = lambda u, a, b, c : -2 * u*(a - b) * (u + h(u,a,b))

dbb = bb(u,a,b,c).diff(b).simplify()
display(dbb)
dbb = dbb.subs(a, b).simplify()
display(dbb)

In [ ]:
dbbn = lambda b, u : 2*u**2*( 1 - (b**2 - 1)*np.real(gs(b)))
plt.plot(b_vals, dbbn(b_vals, .1), label="$\\partial_b\\beta_b$")
plt.xlabel("$b$")
plt.legend()

In [ ]:
# h = lambda u, a, b : (u - b*a) * np.real(gs(b)) - (a - b*u) * np.imag(gs(b))
h = lambda a, b : (1 - b*a) * np.real(gs(b)) - (a - b) * np.imag(gs(b))
h = lambda a, b : (1 + b*a) * np.real(gs(b)) - (a - b) * np.imag(gs(b)) 
u, a = sp.symbols("u a")
h(a, 0)

In [ ]:
d = 3
e = 4 - d

# ba = lambda u, a, b, c : -e*a + 2/(b**2+1) * ((4*b**2 + 6)*a*u + b*(a**2-u**2))

# bu = lambda u, a, b, c : -e*u + 2/(b**2+1) * ((4*b**2 + 5)*u**2 - a**2 + 2*b*u*a)

bu = lambda u, a, b, c : -e*u + 2*u**2/(b**2+1) * ((4*b**2 + 5) - a**2 + 2*b*a)
ba = lambda u, a, b, c : +2*u*(a-b) * (a**2 + 1)/(b**2+1)
bc = lambda u, a, b, c : +4  *  (a - c*u)

bb0 = lambda u, a, b, c : -8/3*u*(a - b*u)

bb = lambda u, a, b, c : -2 * u**2* (a - b) * (1 + h(a,b))


In [ ]:
b = np.linspace(-l, l, 200)
fig, ax = plt.subplots(figsize=(8,5))

# plt.plot(b, bb0(.1, 0, b, 0), 'k')
N = 11
for i,a in enumerate(np.linspace(0, 20, N)): plt.plot(b, h(a, b), color=cm.viridis(i/(N-1)), lw=1, label="$a = {:.3f}$".format(a))
plt.plot(b, 0*b, 'k--')
plt.legend(loc=1, fontsize=12)
ax.set_xlabel("$b$")
ax.set_ylabel("$h(a, b)$")
plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(9,5))

b = np.linspace(-6, l, 500)

N = 11
for i,a in enumerate(np.linspace(0, l, N)): 
    c= cm.viridis(i/(N-1))
    plt.plot(b, bb(.1, a, b, 0), color=c, lw=1, label="$a = {:.3f}$".format(a))
    plt.plot(a, 0, '.', color=c, zorder = 10)


ax.set_xlabel("$b$")
ax.set_ylabel("$b_b(u^*, a, b)$")

plt.plot(b, 0*b, 'k-', lw=.5)

plt.legend(loc=0, fontsize=12)

plt.show()

In [ ]:
import plotly.graph_objects as go

def simulate(v, u0, a0, b0, dt=0.02, nsteps=1000):
    trajectory = np.zeros((nsteps, 3))
    trajectory[0] = [u0, a0, b0]
    for i in range(1, nsteps):
        u, a, b = trajectory[i-1]
        du, da, db = v(u, a, b)
        trajectory[i] = [u + dt*du, a + dt*da, b + dt*db]
    return trajectory

def get_init(N):
    initial_conditions = []
    ar = 10
    br = 4
    ur = .2
    for i in range(N):
        u0 = np.random.rand() * ur
        a0 = (2*np.random.rand() - 1) * ar
        b0 = (2*np.random.rand() - 1) * br
        initial_conditions.append((u0, a0, b0))
    return initial_conditions

# def get_init(N):
#     initial_conditions = []
#     u0 = 0.01
#     for b0 in [0, 1, 2, 4, 8]:
#         for a0 in np.linspace(0, 12, N):
#             initial_conditions.append((u0, a0, b0))
#     return initial_conditions

def plot_flow3D(v, N, fig3D, nsteps=1000):

    initial_conditions = get_init(N)
    for u0, a0, b0 in initial_conditions:
        traj = simulate(v, u0, a0, b0, nsteps=nsteps)
        fig3d.add_trace(go.Scatter3d(
            x=traj[:, 0], y=traj[:, 1], z=traj[:, 2],
            mode="lines", line=dict(width=3), name=f"({u0}, {a0}, {b0})", showlegend=False
        ))
        fig3d.add_trace(go.Scatter3d(
            x=[traj[0,0]], y=[traj[0,1]], z=[traj[0,2]],
            mode='markers', marker=dict(size=5, color='red'), showlegend=False
        ))
        fig3d.add_trace(go.Scatter3d(
            x=[traj[-1,0]], y=[traj[-1,1]], z=[traj[-1,2]],
            mode='markers', marker=dict(size=5, color='blue'), showlegend=False
        ))

    fig3d.update_layout(
        scene=dict(xaxis_title="u",yaxis_title="a",zaxis_title="b",),
        width=1400, height=1400
    )
    for trace in fig3d.data: trace.line.color = 'black'

In [ ]:
f = lambda u, a, b, c : np.array([bu(u,a,b,c), ba(u,a,b,c), bb(u,a,b,c)])
v = lambda u, a, b : - f(u, a, b, 0)

fig3d = go.Figure()
plot_flow3D(v, 200, fig3d, nsteps=2000)
fig3d.show()

In [ ]:
def fillcolors(x1, x2, F, colors, maxlen, steps, dt, sgnx=False, sgny=False):

    for ii in range(x1.shape[0]):
        for jj in range(x1.shape[1]):
            x = x1[ii, jj]
            y = x2[ii, jj]
            for _ in range(steps): 
                dx, dy = F(x, y)
                x += dt * dx
                y += dt * dy

            c = np.sqrt(x**2 + y**2)
            if np.isnan(c): c = maxlen
            s = 1
            if sgnx: s = np.sign(x)
            if sgny: s = np.sign(y)
            colors[ii, jj] = s*min(c, maxlen)
    

In [ ]:
size = 4
steps = 50
N = 100
k1 = 0.2
k2 = 0.3
dt = 0.1
maxlen = np.sqrt(k1**2+k2**2)

x1, x2 = np.linspace(0, k1, N), np.linspace(-k2, k2, N)
x1, x2 = np.meshgrid(x1, x2)

f = lambda u, a, b, c : - np.array([bu(u,a,b,c), ba(u,a,b,c)])

bs = [0, 1., ]
for b in bs:
    colors = np.zeros_like(x1) 
    F = lambda x, y: f(x, y, b, 0)
    fillcolors(x1, x2, F, colors, maxlen, steps, dt)

    fig, ax = plt.subplots(figsize=(size + 2, size), sharex=True, sharey=True)
    u, v = F(x1, x2)
    ax.streamplot(x1, x2, u, v, density=2., linewidth=.6, color='black')

    ax.pcolormesh(
        x1, x2, colors,
        cmap=cm.coolwarm,
        alpha=.7,
        linewidths=0.1,
        rasterized=True
    )

    ax.set_title("$d={:.0f}, \\quad b = {:.3f}$".format(d, b))
    ax.set_xlabel("$u$")
    ax.set_ylabel("$\\alpha_1$")
    ax.set_xlim(0,k1)
    ax.set_ylim(-k2,k2)
 
    # plt.savefig("d={}.svg".format(d))
    plt.show()

In [ ]:
size = 4
steps = 200
N = 50
k1 = 0.3
k2 = 0.9
dt = 0.2
maxlen = np.sqrt(k1**2+k2**2)

x1, x2 = np.linspace(-k1, k1, N), np.linspace(0, k2, N)
x1, x2 = np.meshgrid(x1, x2)

f = lambda u, a, b, c : - np.array([ba(u,a,b,c), bb(u,a,b,c)])

us = [0.0, 0.04, 0.08, 0.1, 0.2]

for U in us:
    colors = np.zeros_like(x1) 
    F = lambda a, b: f(U, a, b, 0.)
    fillcolors(x1, x2, F, colors, maxlen, steps, dt, sgny=True)

    fig, ax = plt.subplots(figsize=(size + 2, size), sharex=True, sharey=True)
    u, v = F(x1, x2)
    ax.streamplot(x1, x2, u, v, density=2., linewidth=.6, color='black')

    ax.pcolormesh(
        x1, x2, colors,
        cmap=cm.coolwarm,
        alpha=.7,
        linewidths=0.1,
        rasterized=True
    )

    ax.set_title("$d={:.0f}, \\quad u = {:.3f}$".format(d, U))
    ax.set_ylabel("$b$") 
    ax.set_xlabel("$\\alpha_1$")
    ax.set_xlim(-k1,k1)
    ax.set_ylim(0,k2)

    plt.show()

In [ ]:
size = 4
N = 100
k1 = .2
k2 = .7
dt = 0.1
steps = 40
maxlen = np.sqrt(k1**2+k2**2)

x1, x2 = np.linspace(-k1, k1, N), np.linspace(-k2, k2, N)
x1, x2 = np.meshgrid(x1, x2)

bs = [-0.5, -0.1, 0, 0.1, 0.5,]
bs = [.4]
U = .1

f = lambda u, a, b, c : - np.array([ba(u,a,b,c), bc(u,a,b,c)])

for b in bs:
    colors = np.zeros_like(x1) 
    F = lambda a, c: f(U, a, b, c)
    print(np.shape(F(x1, x2)))
    fillcolors(x1, x2, F, colors, maxlen, steps, dt, sgny=True)

    fig, ax = plt.subplots(figsize=(size + 2, size), sharex=True, sharey=True)
    u, v = F(x1, x2)
    ax.streamplot(x1, x2, u, v, density=2., linewidth=.6, color='black')

    ax.pcolormesh(
        x1, x2, colors,
        cmap=cm.coolwarm,
        alpha=.7,
        linewidths=0.1,
        rasterized=True
    )

    ax.set_title("$d={:.0f}, \\quad b = {:.3f},\\quad u = {:.3f}$".format(d, b, U))
    ax.set_ylabel("$c$") 
    ax.set_xlabel("$\\alpha_1$")
    ax.set_xlim(-k1,k1)
    ax.set_ylim(-k2,k2)

    plt.show()

In [ ]:
f = lambda u, a, b, c : - np.array([bu(u,a,b,c), ba(u,a,b,c), bb(u,a,b,c), bc(u,a,b,c)])

def simulate_c(u0, a0, b0, c0, dt=0.01, nsteps=500):
    t_values = np.linspace(0, nsteps*dt, nsteps)
    c_values = np.zeros(nsteps)
    c_values[0] = c0
    
    u, a, b, c = u0, a0, b0, c0
    for i in range(1, nsteps):
        du, da, db, dc = f(u, a, b, c)
        u, a, b, c = [u + dt*du, a + dt*da, b + dt*db, c + dt*dc]
        c_values[i] = c
        
    return t_values, c_values


def get_init(N):
    initial_conditions = []
    b0 = 1
    us = [0.2]
    for u0 in us:
        for a1 in np.linspace(-.8, .8, N):
            initial_conditions.append((u0, a1, b0))
    return initial_conditions


fig, ax = plt.subplots(figsize=(6, 3))

initial_conditions = get_init(20)

for u0, a0, b0 in initial_conditions:
    c0 = 0

    t, c_traj = simulate_c(u0, a0, b0, c0, nsteps=1000)
    
    ax.plot(t, c_traj, alpha=0.5, linewidth=1, color='tab:blue')

ax.set_xlabel("$- \\ln b$")
ax.set_ylabel("$c$")
ax.set_title("Evolution of $c$ from $c = 0$ varying $u_0, \\alpha_0, b_0$")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))

for _ in range(40):
    u0, a1, b0 = .1, .1, -.1
    c0 = (2*np.random.rand() - 1) * 4

    t, c_traj = simulate_c(u0, a1, b0, c0, nsteps=1000)
    
    ax.plot(t, c_traj, alpha=0.5, linewidth=1, color='tab:blue')

ax.set_xlabel("$- \\ln b$")
ax.set_ylabel("$c$")
ax.set_title("Evolution of $c$ for random $c_0$ with fixed $u_0, \\alpha_{1, 0}, b_0$")
plt.show()